# Lab Week 10 — Hugging Face Tutorial Lab: `pipeline()`

## 0. Setup

In [ ]:
#!pip install "transformers==4.38.2"
#!pip install torch datasets evaluate "transformers[sentencepiece]" accelerate

#import sys
#print(sys.executable)

/home/user/ai-course/.venv/bin/python


## 1. Transformers, what can they do?

In [2]:
import torch
import transformers

print("Torch:", torch.__version__)
print("Transformers:", transformers.__version__)

Torch: 2.12.0+cu130
Transformers: 4.38.2


#### Note: 
If you use a newer version of the Hugging Face Transformers library, such as Transformers 5.8.1, you may have access to more models and additional pipeline tasks, such as:

* `image-text-to-text`
* `text-to-audio`

However, in the latest version, the `summarization` and `question-answering` pipelines may no longer be available by default.

## Sentiment-Analysis
We didn't assign a specific model for the task, 
So, the default model is ``` distilbert/distilbert-base-uncased-finetuned-sst-2-english ```

In [3]:
from transformers import pipeline

classifier = pipeline("sentiment-analysis")

# Test on multiple sentences
texts = [
    "I've been waiting for a Hugging Face course my whole life.",
    "I hate this so much!"
]

result2 = classifier(texts)
print(result2)

2026-06-08 23:40:28.375921: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:479] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-06-08 23:40:28.472082: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:10575] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-06-08 23:40:28.472718: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1442] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-06-08 23:40:28.619705: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-06-08 23:40:29.926370: W tensorflow/compiler/tf

[{'label': 'POSITIVE', 'score': 0.9982948899269104}, {'label': 'NEGATIVE', 'score': 0.9994558691978455}]


## Zero-Shot Classification

The default model is ``` facebook/bart-large-mnli ```

MNLI: Multi-Genre Natural Language Inference

Zero-shot classification does not magically know every possible label. Instead, Hugging Face converts the classification problem into a natural language inference problem. For each candidate label, it creates a hypothesis such as “This text is about sports.” Then an MNLI model checks whether the original text entails that hypothesis. facebook/bart-large-mnli is used by default because it is a BART model fine-tuned on the MNLI dataset, so it is already good at judging entailment, contradiction, and neutrality.

In [4]:
classifier = pipeline("zero-shot-classification")
classifier(
    "This is a course about the Transformers library",
    candidate_labels=["education", "politics", "business"],
)

No model was supplied, defaulted to facebook/bart-large-mnli and revision c626438 (https://huggingface.co/facebook/bart-large-mnli).
Using a pipeline without specifying a model name and revision in production is not recommended.
/home/user/ai-course/.venv/lib/python3.12/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


{'sequence': 'This is a course about the Transformers library',
 'labels': ['education', 'business', 'politics'],
 'scores': [0.844599187374115, 0.11197410523891449, 0.0434267595410347]}

## Text Generation
Previous defaulted model is: ```openai-community/gpt2 ```

Recent revised default model is  ```HuggingFaceTB/SmolLM3-3B```

In [5]:
generator = pipeline("text-generation", model="gpt2")
generator("In this course, we will teach you how to") #slow - 3B 


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


[{'generated_text': 'In this course, we will teach you how to manage your life.\n\nThis course covers the basics like the life of a professional, and how to manage your finances. You also come up with some general considerations about making things work.\n\n'}]

In [6]:
#Assign more parameters to control the generation

generator = pipeline("text-generation", model="distilgpt2")

outputs = generator(
    "In this course, we will teach you how to",
    max_new_tokens=30,
    truncation=True,
    num_return_sequences=2
)

for i, output in enumerate(outputs, 1):
    print(f"Output {i}:")
    print(output["generated_text"])
    print()

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Output 1:
In this course, we will teach you how to run Python on a system-wide computer. The course will cover how to run the Python programming language with Python 1.x and 2.x.

Output 2:
In this course, we will teach you how to use a technique that you can use to understand how to integrate an existing tool in your own application and how to use it in your other projects. How



## Fill Mask

Default model is ``` distilbert/distilroberta-base```

In [7]:
unmasker = pipeline("fill-mask")
unmasker("This course will teach you all about <mask> models.", top_k=2)

No model was supplied, defaulted to distilbert/distilroberta-base and revision ec58a5b (https://huggingface.co/distilbert/distilroberta-base).
Using a pipeline without specifying a model name and revision in production is not recommended.
Some weights of the model checkpoint at distilbert/distilroberta-base were not used when initializing RobertaForMaskedLM: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


[{'score': 0.19619761407375336,
  'token': 30412,
  'token_str': ' mathematical',
  'sequence': 'This course will teach you all about mathematical models.'},
 {'score': 0.04052722454071045,
  'token': 38163,
  'token_str': ' computational',
  'sequence': 'This course will teach you all about computational models.'}]

## NER: Named Entity Recognition
Aggregate token-level predictions into word/entity-level predictions using a simple grouping rule.


In [8]:
ner = pipeline("ner", aggregation_strategy="simple")
ner("My name is Peiyuan and I work at Conestoga College in Waterloo.")

No model was supplied, defaulted to dbmdz/bert-large-cased-finetuned-conll03-english and revision f2482bf (https://huggingface.co/dbmdz/bert-large-cased-finetuned-conll03-english).
Using a pipeline without specifying a model name and revision in production is not recommended.
Some weights of the model checkpoint at dbmdz/bert-large-cased-finetuned-conll03-english were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


[{'entity_group': 'PER',
  'score': 0.9846199,
  'word': 'Peiyuan',
  'start': 11,
  'end': 18},
 {'entity_group': 'ORG',
  'score': 0.9766252,
  'word': 'Conestoga College',
  'start': 33,
  'end': 50},
 {'entity_group': 'LOC',
  'score': 0.9702344,
  'word': 'Waterloo',
  'start': 54,
  'end': 62}]

## Question-Answering
Default model: ```distilbert/distilbert-base-cased-distilled-squad```

In [9]:
from transformers import pipeline

question_answerer = pipeline("question-answering")
question_answerer(
    question="Where do I work?",
    context="My name is Sylvain and I work at Hugging Face in Brooklyn",
)

No model was supplied, defaulted to distilbert/distilbert-base-cased-distilled-squad and revision 626af31 (https://huggingface.co/distilbert/distilbert-base-cased-distilled-squad).
Using a pipeline without specifying a model name and revision in production is not recommended.


{'score': 0.6949764490127563, 'start': 33, 'end': 45, 'answer': 'Hugging Face'}

In [10]:
summarizer = pipeline("summarization")
summarizer(
    """
    America has changed dramatically during recent years. Not only has the number of 
    graduates in traditional engineering disciplines such as mechanical, civil, 
    electrical, chemical, and aeronautical engineering declined, but in most of 
    the premier American universities engineering curricula now concentrate on 
    and encourage largely the study of engineering science. As a result, there 
    are declining offerings in engineering subjects dealing with infrastructure, 
    the environment, and related issues, and greater concentration on high 
    technology subjects, largely supporting increasingly complex scientific 
    developments. While the latter is important, it should not be at the expense 
    of more traditional engineering.

    Rapidly developing economies such as China and India, as well as other 
    industrial countries in Europe and Asia, continue to encourage and advance 
    the teaching of engineering. Both China and India, respectively, graduate 
    six and eight times as many traditional engineers as does the United States. 
    Other industrial countries at minimum maintain their output, while America 
    suffers an increasingly serious decline in the number of engineering graduates 
    and a lack of well-educated engineers.
"""
)

No model was supplied, defaulted to sshleifer/distilbart-cnn-12-6 and revision a4f8f3e (https://huggingface.co/sshleifer/distilbart-cnn-12-6).
Using a pipeline without specifying a model name and revision in production is not recommended.


[{'summary_text': ' America has changed dramatically during recent years . The number of engineering graduates in the U.S. has declined in traditional engineering disciplines such as mechanical, civil,    electrical, chemical, and aeronautical engineering . Rapidly developing economies such as China and India continue to encourage and advance the teaching of engineering .'}]

In [11]:
translator = pipeline("translation", model="Helsinki-NLP/opus-mt-fr-en")
translator("Ce cours est produit par Hugging Face.")

/home/user/ai-course/.venv/lib/python3.12/site-packages/transformers/models/marian/tokenization_marian.py:197: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


[{'translation_text': 'This course is produced by Hugging Face.'}]

## 2. What behind `pipeline()`? 
## The simplest usage: `pipeline("sentiment-analysis")`

The Hugging Face `pipeline()` API hides three steps:

```text
1. Preprocessing: raw text → tokens → token IDs → tensors
2. Model inference: tensors → Transformer → classification head → logits
3. Postprocessing: logits → softmax probabilities → labels and scores
```

The default sentiment-analysis checkpoint used in the Hugging Face tutorial is:

```python
distilbert-base-uncased-finetuned-sst-2-english
```
A **checkpoint** saved pretrained/fine-tuned model.

In [12]:
from transformers import pipeline

classifier = pipeline("sentiment-analysis")
# Test bias in the model: "Muted" may often be associated with negative sentiment

texts = [
    "I've been waiting for a Hugging Face course my whole life.",
    "I hate this so much!",
    "The movie was okay, but not very exciting.",
    "The movie has a muted visual style.",
]

results = classifier(texts)

for text, result in zip(texts, results):
    print("Text:", text)
    print("Result:", result)
    print("-" * 80)


No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision af0f99b (https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english).
Using a pipeline without specifying a model name and revision in production is not recommended.


Text: I've been waiting for a Hugging Face course my whole life.
Result: {'label': 'POSITIVE', 'score': 0.9982948899269104}
--------------------------------------------------------------------------------
Text: I hate this so much!
Result: {'label': 'NEGATIVE', 'score': 0.9994558691978455}
--------------------------------------------------------------------------------
Text: The movie was okay, but not very exciting.
Result: {'label': 'NEGATIVE', 'score': 0.9978721141815186}
--------------------------------------------------------------------------------
Text: The movie has a muted visual style.
Result: {'label': 'NEGATIVE', 'score': 0.9978736639022827}
--------------------------------------------------------------------------------


### Step 1: Preprocessing - Tokenization
The default model is ```distilbert/distilbert-base-uncased-finetuned-sst-2-english```
You can choose a spedific model

In [13]:
checkpoint = "distilbert-base-uncased-finetuned-sst-2-english"
print(checkpoint)

distilbert-base-uncased-finetuned-sst-2-english


In [14]:
# [CLS]=101(classification); [SEP]=102(end of sentence); [PAD]=0; [UNK]=100; [MASK]=103
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(checkpoint)

raw_inputs = [
    "I've been waiting for a Hugging Face course my whole life.",
    "I hate this so much!",
]

inputs = tokenizer(
    raw_inputs,
    padding=True,
    truncation=True,
    return_tensors="pt"
)

#### What can Tokenizer do? 
When we pass raw text into a tokenizer, the tokenizer converts the text into a format that a Transformer model can understand. The model cannot directly read English words. It needs numbers.

A tokenizer usually outputs two important things:

1. `input_ids` are token IDs. For example, a word or subword is mapped to an integer.
2. `attention_mask` tells the model which positions are real tokens and which positions are padding.
    Usually:
    ```text
    1 = real token
    0 = padding token
    ```


#### Notice special tokens
For BERT-style models:
- `[CLS]` is often used as the sentence-level representation.
- `[SEP]` marks the end of a sentence.
- `[PAD]` is added to make sequences in the same batch have the same length.

Play with tokenizer in: https://huggingface.co/spaces/Xenova/the-tokenizer-playground

In [15]:
print(inputs)
print("input_ids shape:", inputs["input_ids"].shape)
print("attention_mask shape:", inputs["attention_mask"].shape)

print("\ninput_ids:")
print(inputs["input_ids"])

print("\nattention_mask:")
print(inputs["attention_mask"])

{'input_ids': tensor([[  101,  1045,  1005,  2310,  2042,  3403,  2005,  1037, 17662,  2227,
          2607,  2026,  2878,  2166,  1012,   102],
        [  101,  1045,  5223,  2023,  2061,  2172,   999,   102,     0,     0,
             0,     0,     0,     0,     0,     0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0]])}
input_ids shape: torch.Size([2, 16])
attention_mask shape: torch.Size([2, 16])

input_ids:
tensor([[  101,  1045,  1005,  2310,  2042,  3403,  2005,  1037, 17662,  2227,
          2607,  2026,  2878,  2166,  1012,   102],
        [  101,  1045,  5223,  2023,  2061,  2172,   999,   102,     0,     0,
             0,     0,     0,     0,     0,     0]])

attention_mask:
tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0]])


In [16]:
for i, sentence_ids in enumerate(inputs["input_ids"]):
    tokens = tokenizer.convert_ids_to_tokens(sentence_ids)
    print(f"Sentence {i + 1}:")
    print(tokens)
    print()

Sentence 1:
['[CLS]', 'i', "'", 've', 'been', 'waiting', 'for', 'a', 'hugging', 'face', 'course', 'my', 'whole', 'life', '.', '[SEP]']

Sentence 2:
['[CLS]', 'i', 'hate', 'this', 'so', 'much', '!', '[SEP]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]']



### Step 2: Model inference
### *2.1 Pass inputs through the base model*

First, we use `AutoModel`.
`AutoModel` loads the base Transformer model **without the classification head**.
It outputs hidden states/features, not final sentiment labels.


In [17]:
from transformers import AutoModel

base_model = AutoModel.from_pretrained(checkpoint)
base_outputs = base_model(**inputs)

print(base_outputs.last_hidden_state.shape)


torch.Size([2, 16, 768])


#### Understanding the shape

The output shape is usually:  [batch_size, sequence_length, hidden_size]

For this example, [2, 16, 768]:
- `2`: two input sentences
- `16`: token length after padding
- `768`: hidden vector dimension for each token

This is not yet a classification result.

### *2.2 Use a model with a classification head*

```AutoModel``` does choose the correct base model architecture from the checkpoint, but it only loads the base Transformer body. 

If you want sentiment classification, you need ```AutoModelForSequenceClassification ```, which adds a task-specific classification head on top of the base Transformer.

In [18]:
from transformers import AutoModelForSequenceClassification

classification_model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)
outputs = classification_model(**inputs)

print(outputs.logits)
print("logits shape:", outputs.logits.shape)


tensor([[-3.1071,  3.2654],
        [ 4.1692, -3.3464]], grad_fn=<AddmmBackward0>)
logits shape: torch.Size([2, 2])


#### What are the outputs?
`logits` are the raw, unnormalized scores produced by the final layer of the model. They are **not probabilities**.

For sentiment analysis with two labels, each sentence gets two logits: ```[score_for_NEGATIVE, score_for_POSITIVE]```
    So the shape is: [batch_size, number_of_labels], 
        e.g.[2, 2]: There are 2 sentences into a 2-class sentiment model.

**output**: {[-3.1071,  3.2654],[4.1692, -3.3464]} 
1. in the first sentence,"I've been waiting for a Hugging Face course my whole life.", negative=-3.1071, positive=3.2654
2. and the second sentence, "I hate this so much!", negative=4.1692, positive =-3.3464

In [19]:
print("id2label mapping:")
print(classification_model.config.id2label)

print("\nlabel2id mapping:")
print(classification_model.config.label2id)

id2label mapping:
{0: 'NEGATIVE', 1: 'POSITIVE'}

label2id mapping:
{'NEGATIVE': 0, 'POSITIVE': 1}


### Step 3: Post Analysis - Convert logits to probabilities with softmax

The model output logits must be converted into probabilities.
We use:

```python
softmax(logits)
```
The largest logit usually becomes the largest probability.


In [20]:
import torch

probabilities = torch.nn.functional.softmax(outputs.logits, dim=-1)
print(probabilities)


tensor([[1.7051e-03, 9.9829e-01],
        [9.9946e-01, 5.4418e-04]], grad_fn=<SoftmaxBackward0>)


### Convert probabilities into labels
Now we manually reproduce the final output of the pipeline.

In [21]:
predicted_class_ids = torch.argmax(probabilities, dim=-1)

for i, text in enumerate(raw_inputs):
    class_id = predicted_class_ids[i].item()
    label = classification_model.config.id2label[class_id]
    score = probabilities[i][class_id].item()

    print("Text:", text)
    print("Predicted label:", label)
    print("Score:", score)
    print("-" * 80)


Text: I've been waiting for a Hugging Face course my whole life.
Predicted label: POSITIVE
Score: 0.9982948899269104
--------------------------------------------------------------------------------
Text: I hate this so much!
Predicted label: NEGATIVE
Score: 0.9994558691978455
--------------------------------------------------------------------------------


### Compare manual result with `pipeline()`

Now we check whether our manual process gives the same meaning as the high-level pipeline.


In [22]:
classifier = pipeline("sentiment-analysis", model=checkpoint, tokenizer=checkpoint)

pipeline_results = classifier(raw_inputs)

for text, result in zip(raw_inputs, pipeline_results):
    print("Text:", text)
    print("Pipeline result:", result)
    print("-" * 80)


Text: I've been waiting for a Hugging Face course my whole life.
Pipeline result: {'label': 'POSITIVE', 'score': 0.9982948899269104}
--------------------------------------------------------------------------------
Text: I hate this so much!
Pipeline result: {'label': 'NEGATIVE', 'score': 0.9994558691978455}
--------------------------------------------------------------------------------


In [23]:
from transformers import pipeline

classifier = pipeline("sentiment-analysis")

# Test on multiple sentences
texts = [
    "I've been waiting for a Hugging Face course my whole life.",
    "I hate this so much!"
]

result = classifier(texts)
print(result)

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision af0f99b (https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english).
Using a pipeline without specifying a model name and revision in production is not recommended.


[{'label': 'POSITIVE', 'score': 0.9982948899269104}, {'label': 'NEGATIVE', 'score': 0.9994558691978455}]


### Try your own examples
Can you modify the following texts and observe the output.

In [24]:
student_texts = [
    "This course is difficult but very useful.",
    "The homework is confusing and frustrating.",
    "I am not sure whether I like this movie.",
    "The product is not bad at all.",
]

student_results = classifier(student_texts)

for text, result in zip(student_texts, student_results):
    print("Text:", text)
    print("Result:", result)
    print("-" * 80)


Text: This course is difficult but very useful.
Result: {'label': 'POSITIVE', 'score': 0.9954234957695007}
--------------------------------------------------------------------------------
Text: The homework is confusing and frustrating.
Result: {'label': 'NEGATIVE', 'score': 0.9991816878318787}
--------------------------------------------------------------------------------
Text: I am not sure whether I like this movie.
Result: {'label': 'NEGATIVE', 'score': 0.9955669045448303}
--------------------------------------------------------------------------------
Text: The product is not bad at all.
Result: {'label': 'POSITIVE', 'score': 0.998784601688385}
--------------------------------------------------------------------------------


### Limitation: sentence meaning can be subtle

A sentiment model may struggle with:

- sarcasm
- mixed opinions
- negation
- domain-specific language
- long context

For example:

```text
This movie is so good that I almost fell asleep.
```

The surface words may look positive, but the actual meaning is negative/sarcastic.


In [25]:
tricky_texts = [
    "This movie is so good that I almost fell asleep.",
    "The phone is cheap, but surprisingly reliable.",
    "I expected to hate it, but I actually loved it.",
    "Not bad at all.",
]

tricky_results = classifier(tricky_texts)

for text, result in zip(tricky_texts, tricky_results):
    print("Text:", text)
    print("Result:", result)
    print("-" * 80)


Text: This movie is so good that I almost fell asleep.
Result: {'label': 'POSITIVE', 'score': 0.9998125433921814}
--------------------------------------------------------------------------------
Text: The phone is cheap, but surprisingly reliable.
Result: {'label': 'POSITIVE', 'score': 0.9980798959732056}
--------------------------------------------------------------------------------
Text: I expected to hate it, but I actually loved it.
Result: {'label': 'POSITIVE', 'score': 0.9998137354850769}
--------------------------------------------------------------------------------
Text: Not bad at all.
Result: {'label': 'POSITIVE', 'score': 0.99928218126297}
--------------------------------------------------------------------------------
